In [1]:
import tensorflow_datasets as tfds
dataset, info = tfds.load("imdb_reviews",split=["train[:10%]", "test[:10%]"], with_info=True, as_supervised=True)

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dl Completed...: 100%|██████████| 1/1 [04:40<00:00, 280.03s/ url]


Dataset imdb_reviews downloaded and prepared to C:\Users\hp\tensorflow_datasets\imdb_reviews\plain_text\1.0.0. Subsequent calls will reuse this data.


In [3]:
train_dataset, test_dataset = dataset
train_dataset, test_dataset

(<_PrefetchDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>,
 <_PrefetchDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>)

In [6]:
info.features

FeaturesDict({
    'label': ClassLabel(shape=(), dtype=int64, num_classes=2),
    'text': Text(shape=(), dtype=string),
})

In [7]:
for text,label in train_dataset.take(3):
    print("Text: ", text.numpy())
    print("Label: ", label.numpy())
    print("-" * 50)

Text:  b"This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it."
Label:  0
--------------------------------------------------
Text:  b'I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on this oc

In [8]:
for text,label in train_dataset.take(3):
    print("Text: ", text.numpy().decode("utf-8"))
    print("Label: ", label.numpy())
    print("-" * 50)

Text:  This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it.
Label:  0
--------------------------------------------------
Text:  I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on this occasio

In [ ]:
import tensorflow as tf
vocab_size = 10000
tokenizer=tf.keras.layers.TextVectorization(
    max_tokens=vocab_size,
    output_sequence_length=200
)

# A tokenizer converts text → numbers (because ML models only understand numbers).

In [10]:
text = ["This movie is amazing"]

tokenizer.adapt(text)

output = tokenizer(text)
print(output)

tf.Tensor(
[[2 3 4 5 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]], shape=(1, 200), dtype=int64)


In [13]:
train_text=train_dataset.map(lambda x,y: x)
tokenizer.adapt(train_text)


In [14]:
def preprocess(x,y):
    return tokenizer(x),y

train_data = train_dataset.map(preprocess).batch(32)
test_data = test_dataset.map(preprocess).batch(32)

In [15]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 16),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [16]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [17]:
model.fit(train_data, epochs=5, validation_data=test_data)

Epoch 1/5
79/79 [==============================] - 8s 56ms/step - loss: 0.6920 - accuracy: 0.5668 - val_loss: 0.6906 - val_accuracy: 0.5216
Epoch 2/5
79/79 [==============================] - 4s 52ms/step - loss: 0.6816 - accuracy: 0.6812 - val_loss: 0.6741 - val_accuracy: 0.6756
Epoch 3/5
79/79 [==============================] - 3s 39ms/step - loss: 0.6383 - accuracy: 0.7896 - val_loss: 0.6253 - val_accuracy: 0.7640
Epoch 4/5
79/79 [==============================] - 5s 61ms/step - loss: 0.5464 - accuracy: 0.8524 - val_loss: 0.5508 - val_accuracy: 0.7944
Epoch 5/5
79/79 [==============================] - 4s 51ms/step - loss: 0.4323 - accuracy: 0.8920 - val_loss: 0.4805 - val_accuracy: 0.8136


In [18]:
model.evaluate(test_data)

79/79 [==============================] - 2s 18ms/step - loss: 0.4805 - accuracy: 0.8136


[0.48051512241363525, 0.8136000037193298]

In [19]:
sample = ["This movie was fantastic!"]

pred = model.predict(tokenizer(sample))
print(pred)

1/1 [==============================] - 0s 231ms/step
[[0.5143931]]


In [ ]:
# Load Data → See Data → Tokenize → Convert → Train → Test